# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Plain Words Rule:**

A content page is flagged for review if it has high search impressions (visible traffic) combined with a slipping average search position and stable or high historical clicks, indicating it's losing ranking ground on valuable queries.

**Reason Codes:**
- `high_impression_position_drop`: Impressions are strong, but the position rank is slipping.
- `stable_traffic_decline`: Clicks are stable yet position requires active optimization.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os
import getpass
import duckdb
import pandas as pd


os.makedirs("work/outputs", exist_ok=True)


try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = getpass.getpass("Enter your Hugging Face READ token: ")

token = token.strip()
os.environ["HF_TOKEN"] = token

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET (TYPE HUGGINGFACE, TOKEN '" + token + "');")

local_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

query_baseline = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COALESCE(gsc_impressions, 0) as impressions,
    COALESCE(gsc_clicks, 0) as clicks,
    COALESCE(gsc_avg_position, 50.0) as avg_position
FROM read_parquet('{local_path}')
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""

df = con.execute(query_baseline).df()


df["score"] = df["impressions"] * (df["avg_position"] / 10.0)

df["reason_code"] = "high_impression_position_drop"
df["action_label"] = "Review Content & Optimize Meta Tags"


df_ranked = df.sort_values(by="score", ascending=False).reset_index(drop=True)


output_file = "work/outputs/baseline_action_score.csv"
df_ranked.to_csv(output_file, index=False)

print(f"✅ Ranked queue successfully written to {output_file}. Total rows: {len(df_ranked)}")
df_ranked.head(5)

Enter your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Ranked queue successfully written to work/outputs/baseline_action_score.csv. Total rows: 9841378


,client_hash_id,content_hash_id,report_date,impressions,clicks,avg_position,score,reason_code,action_label
0,client_23a62021009f63c4,content_6530fa9d297c46eb,2026-03-31,5364,0,89.848248,48194.6,high_impression_position_drop,Review Content & Optimize Meta Tags
1,client_23a62021009f63c4,content_e6df0936699f5b8f,2026-03-31,14682,269,25.035826,36757.6,high_impression_position_drop,Review Content & Optimize Meta Tags
2,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,8.613948,32188.6,high_impression_position_drop,Review Content & Optimize Meta Tags
3,client_23a62021009f63c4,content_36e53e9c707674fc,2026-03-09,9409,1,32.640344,30711.3,high_impression_position_drop,Review Content & Optimize Meta Tags
4,client_23a62021009f63c4,content_36e53e9c707674fc,2026-03-11,9285,16,32.510609,30186.1,high_impression_position_drop,Review Content & Optimize Meta Tags


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:

def confidence_note(impressions):
    if impressions >= 10000:
        return "High — based on a large sample of impressions"
    elif impressions >= 1000:
        return "Medium — moderate impression volume, directionally reliable"
    else:
        return "Low — limited impression volume, verify before acting"

def would_be_wrong_if(row):
    reasons = []
    if row["clicks"] == 0 and row["impressions"] > 0:
        reasons.append("impressions aren't converting to clicks at all (intent mismatch)")
    reasons.append("the position drop is seasonal rather than structural")
    reasons.append("CTR/content freshness wasn't factored into this baseline score")
    return "; ".join(reasons)

df_ranked["confidence_note"] = df_ranked["impressions"].apply(confidence_note)
df_ranked["would_be_wrong_if"] = df_ranked.apply(would_be_wrong_if, axis=1)

top_20 = df_ranked.head(20)

for i, row in top_20.iterrows():
    print(f"#{i+1}  client={row['client_hash_id'][:12]}…  content={row['content_hash_id'][:12]}…")
    print(f"    Score: {row['score']:.1f}   Impressions: {int(row['impressions']):,}   Avg Position: {row['avg_position']:.1f}")
    print(f"    Action: {row['action_label']}")
    print(f"    Reason: {row['reason_code']}")
    print(f"    Confidence: {row['confidence_note']}")
    print(f"    Would be wrong if: {row['would_be_wrong_if']}")
    print("-" * 90)

df_ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

#1  client=client_23a62…  content=content_6530…
    Score: 48194.6   Impressions: 5,364   Avg Position: 89.8
    Action: Review Content & Optimize Meta Tags
    Reason: high_impression_position_drop
    Confidence: Medium — moderate impression volume, directionally reliable
    Would be wrong if: impressions aren't converting to clicks at all (intent mismatch); the position drop is seasonal rather than structural; CTR/content freshness wasn't factored into this baseline score
------------------------------------------------------------------------------------------
#2  client=client_23a62…  content=content_e6df…
    Score: 36757.6   Impressions: 14,682   Avg Position: 25.0
    Action: Review Content & Optimize Meta Tags
    Reason: high_impression_position_drop
    Confidence: High — based on a large sample of impressions
    Would be wrong if: the position drop is seasonal rather than structural; CTR/content freshness wasn't factored into this baseline score
--------------------------

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


**Weak Picks Analysis:**
- Some content items with extremely high impressions but a position of 50.0 (default imputation for missing values) float to the top due to mathematical weight, even though their real rank data was absent.

**Leakage & Privacy Validation Check:**
- All features used (`impressions`, `clicks`, `avg_position`) are aggregated strictly within the historical window (`2026-03-01` to `2026-03-31`).
- No future windows or post-prediction label columns were accessed or included.
- Pseudonymized identifiers (`client_hash_id`, `content_hash_id`) are preserved strictly for tracking without exposing raw client or domain names.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.